In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset,ConcatDataset
import numpy as np
import pandas as pd

In [35]:
train_base="../dataset/train/"
val_base="../dataset/val/"
train_paths=[]
val_paths=[]
for i in range(0,8):
    filename=f"ch{i}.csv"
    train_filepath=train_base+"train_"+filename
    val_filepath=val_base+"val_"+filename
    print(train_filepath)
    print(val_filepath)
    train_paths.append(train_filepath)
    val_paths.append(val_filepath)

../dataset/train/train_ch0.csv
../dataset/val/val_ch0.csv
../dataset/train/train_ch1.csv
../dataset/val/val_ch1.csv
../dataset/train/train_ch2.csv
../dataset/val/val_ch2.csv
../dataset/train/train_ch3.csv
../dataset/val/val_ch3.csv
../dataset/train/train_ch4.csv
../dataset/val/val_ch4.csv
../dataset/train/train_ch5.csv
../dataset/val/val_ch5.csv
../dataset/train/train_ch6.csv
../dataset/val/val_ch6.csv
../dataset/train/train_ch7.csv
../dataset/val/val_ch7.csv


In [3]:
class CrudeDataset(Dataset):

    mapping = {f"ch{i}": i for i in range(8)}

    def __init__(self, path, seq_len=100):
        self.df = pd.read_csv(path)

        # Ensure correct types early
        self.df["channel"] = self.df["channel"].astype(str)

        # Pre-group for speed + stability
        self.groups = self.df.groupby("batch_id")

        # Keep only valid batch_ids (length >= 2)
        self.batch_ids = [
            bid for bid, g in self.groups
            if len(g) >= 2
        ]

        self.seq_len = seq_len

    def __len__(self):
        return len(self.batch_ids)

    def __getitem__(self, idx):
        batch_id = self.batch_ids[idx]

        # ✅ Safe group access (no copy warning)
        batch_data = self.groups.get_group(batch_id).sort_values("time").copy()

        # =========================
        # Channel mapping (SAFE)
        # =========================
        batch_data["channel"] = batch_data["channel"].map(self.mapping)

        # Fill unmapped channels (important!)
        batch_data["channel"] = batch_data["channel"].fillna(0)

        # =========================
        # TARGET (spoofed)
        # =========================
        target = torch.tensor(
            [float(batch_data.iloc[-1]["spoofed"])],
            dtype=torch.float32
        )

        # =========================
        # DROP UNUSED
        # =========================
        batch_data = batch_data.drop(columns=["spoofed", "time", "batch_id"])

        # Convert to numpy safely
        data = batch_data.to_numpy(dtype=np.float32)

        # =========================
        # SPLIT
        # =========================
        features = data[:-1]           # (T-1, F)
        transformed_target = data[-1]  # (F,)

        # =========================
        # FIX VARIABLE LENGTH (IMPORTANT)
        # =========================
        if len(features) < self.seq_len:
            pad = np.zeros((self.seq_len - len(features), features.shape[1]), dtype=np.float32)
            features = np.vstack([pad, features])
        else:
            features = features[-self.seq_len:]

        # =========================
        # TO TENSOR + CLEAN
        # =========================
        features = torch.from_numpy(features)
        transformed_target = torch.from_numpy(transformed_target)

        # Numerical safety
        features = torch.nan_to_num(features, nan=0.0, posinf=10.0, neginf=-10.0)
        transformed_target = torch.nan_to_num(transformed_target, nan=0.0, posinf=10.0, neginf=-10.0)

        # Clamp (stabilizes training)
        features = torch.clamp(features, -5.0, 5.0)
        transformed_target = torch.clamp(transformed_target, -5.0, 5.0)

        return target, features, transformed_target

In [4]:
combined_val_dataset = []
combined_train_dataset = []

for train_paths, val_paths in zip(train_paths, val_paths):
    train_dataset=CrudeDataset(train_paths)
    val_dataset=CrudeDataset(val_paths)
    combined_train_dataset.append(train_dataset)
    combined_val_dataset.append(val_dataset)

In [5]:
combined_val_dataset = ConcatDataset(combined_val_dataset)
combined_train_dataset = ConcatDataset(combined_train_dataset)

In [6]:
train_loader = DataLoader(combined_train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(combined_val_dataset, batch_size=32, shuffle=False)

In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim=14, d_model=128, nhead=8, num_layers=3, 
                 dim_feedforward=256, dropout=0.1,):
        super(TimeSeriesTransformer, self).__init__()
        
        self.d_model = d_model
                
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
                
        # Output layers
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Prediction head
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, input_dim)
        )
        
    def forward(self, x):
        batch_size, seq_len, input_dim = x.shape
        # Reshape for transformer
        outputs= {
            'input': x,
            'projected': None,
            'positional_encoded': None,
            'encoder_outputs': [],
            "before_output_projection": None,
            'final_output': None
        }
        x = x.view(batch_size , seq_len, input_dim)
        
        # Project input
        x = self.input_projection(x) * math.sqrt(self.d_model)
        outputs['projected'] = x.detach().cpu()
        # Add positional encoding
        x = self.pos_encoder(x)
        outputs['positional_encoded'] = x.detach().cpu()
        
        residual = x[:, -1, :]  # (batch, input_dim)

        # Apply transformer
        x = self.transformer(x)  # (batch, seq_len, d_model)
        outputs['encoder_outputs'] = x.detach().cpu()
        
        # Take last timestep
        x = x[:, -1, :]  # (batch, d_model)
        
        # Reshape back
        x = x.view(batch_size, self.d_model)
        
        # Apply layer norm and residual
        x = self.layer_norm1(x+residual)
        x = self.dropout(x)

        outputs["before_output_projection"] = x.detach().cpu()
        
        # Output projection
        output = self.output_projection(x)
        outputs['final_output'] = output.detach().cpu()
        
        return output, outputs

class CNNBlock(nn.Module):
    """CNN block for binary classification after transformer output"""
    def __init__(self, in_channels, hidden_channels=256, dropout=0.1):
        super(CNNBlock, self).__init__()
        
        self.conv_layers = nn.Sequential(
            # First conv block
            nn.Conv2d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            
            # Second conv block
            nn.Conv2d(hidden_channels, hidden_channels // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_channels // 2),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            
            # Third conv block
            nn.Conv2d(hidden_channels // 2, hidden_channels // 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_channels // 4),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
        )
        
        # Global average pooling and classification head
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels // 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 1)  # Binary classification output
        )
        
    def forward(self, x):
        # x shape: (batch, channels=nhead*num_layers, seq_len, d_model)
        x = self.conv_layers(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)  # Flatten
        cnn_features = x  # Save features for potential use
        x = self.classifier(x)
        return x, cnn_features


class ImageAnalysis(nn.Module):

    def __init__(self, input_dim=14, d_model=128, nhead=8, num_layers=3, 
                 dim_feedforward=256, dropout=0.1, cnn_hidden=256):
        super(ImageAnalysis, self).__init__()

        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers

        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # Transformer encoder
        self.encoder_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                batch_first=True
            )
            for _ in range(num_layers)
        ])

        self.to_image = nn.Linear(d_model, nhead * d_model)
        
        # CNN block for binary classification
        # Input channels = nhead * num_layers (concatenated across transformer layers)
        self.cnn_block = CNNBlock(
            in_channels=nhead * num_layers,  # FIXED: multiplied by num_layers
            hidden_channels=cnn_hidden,
            dropout=dropout
        )

    def forward(self, x):
        batch_size, seq_len, input_dim = x.shape

        images=[]

        x = x.view(batch_size, seq_len, input_dim)

        # Project input
        x = self.input_projection(x) * math.sqrt(self.d_model)

        # Add positional encoding
        x = self.pos_encoder(x)

        # Apply transformer
        for layer_idx, encoder_layer in enumerate(self.encoder_layers):
            x = encoder_layer(x)

            image = self.to_image(x)  # (batch, seq_len, nhead*d_model)
            image = image.view(batch_size, seq_len, self.nhead, self.d_model)
            image = image.permute(0, 2, 1, 3)  # (batch, nhead, seq_len, d_model)
            images.append(image)

        # Concatenate all transformer layer outputs
        # Shape: (batch, nhead * num_layers, seq_len, d_model)
        result = torch.cat(images, dim=1)
        
        # Apply CNN block for binary classification
        cnn_output, cnn_features = self.cnn_block(result)
        
        final_output = torch.sigmoid(cnn_output)
        
        return final_output, cnn_features


In [8]:
# Initialize models
timeseries_model = TimeSeriesTransformer()
image_analysis_model = ImageAnalysis(
    input_dim=14,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.3, 
    cnn_hidden=128
)

In [9]:
from tqdm import tqdm
import numpy as np
from matplotlib import pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [10]:

checkpoint = torch.load('finetuned_with_head.pth', map_location=device)
timeseries_model.load_state_dict(checkpoint['timeseries_model'])
image_checkpoint = torch.load('best_cnn_model.pth', map_location=device)
image_analysis_model.load_state_dict(image_checkpoint)


<All keys matched successfully>

In [11]:

timeseries_model = timeseries_model.to(device)
image_analysis_model = image_analysis_model.to(device)


In [14]:
timeseries_model.eval()

X_test_transf_block = []
y_test_transf_block = []

X_test_image_block = []

with torch.no_grad():
    for target, features, transformed_target in val_loader:

        features = features.to(device)
        transformed_target = transformed_target.to(device)

        # Extract representation
        ts_output, outputs = timeseries_model(features)   # (B, 15)

        image_input = torch.cat([features, transformed_target.unsqueeze(1)], dim=1)  # (batch, seq_len, num_features + target_dim)
        image_input = image_input.to(device)
        y_pred,features = image_analysis_model(image_input)

        temp = outputs["before_output_projection"]
        temp = torch.tensor(temp, dtype=torch.float32).to(device)

        # Choose fusion strategy
        fusion = torch.cat([ts_output, transformed_target.to(device), temp], dim=-1)  # (B, 28)
        # Move to CPU numpy
        X_test_transf_block.append(fusion.cpu().numpy())
        y_test_transf_block.append(target.numpy())

        temp = torch.cat([transformed_target, features], dim=1)
        X_test_image_block.append(temp.detach().cpu().numpy())

X_val_tranf = np.concatenate(X_test_transf_block, axis=0)
X_val_image = np.concatenate(X_test_image_block, axis=0)
y_val = np.concatenate(y_test_transf_block, axis=0).reshape(-1)

print("Transfoemer Feature shape:", X_val_tranf.shape)  # (N, 15)
print("CNN Features shape:", X_val_image.shape)
print("Target shape:", y_val.shape)

/tmp/ipykernel_31438/1417880857.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  temp = torch.tensor(temp, dtype=torch.float32).to(device)


Transfoemer Feature shape: (27532, 156)
CNN Features shape: (27532, 46)
Target shape: (27532,)


In [ ]:
timeseries_model.eval()

X_train_transf_block = []
y_train_transf_block = []
X_train_image_block = []

with torch.no_grad():
    for target, features, transformed_target in train_loader:

        features = features.to(device)
        transformed_target = transformed_target.to(device)

        # Extract representation
        ts_output, outputs = timeseries_model(features)   # (B, 15)

        image_input = torch.cat([features, transformed_target.unsqueeze(1)], dim=1)  # (batch, seq_len, num_features + target_dim)
        image_input = image_input.to(device)
        y_pred,features = image_analysis_model(image_input)

        temp = outputs["before_output_projection"]
        temp = torch.tensor(temp, dtype=torch.float32).to(device)

        # Choose fusion strategy
        fusion = torch.cat([ts_output, transformed_target.to(device), temp], dim=-1)  # (B, 28)
        # Move to CPU numpy
        X_train_transf_block.append(fusion.cpu().numpy())
        y_train_transf_block.append(target.numpy())

        temp = torch.cat([transformed_target, features], dim=1)
        X_train_image_block.append(temp.detach().cpu().numpy())

X_train_tranf = np.concatenate(X_train_transf_block, axis=0)
X_train_image = np.concatenate(X_train_image_block, axis=0)
y_train = np.concatenate(y_train_transf_block, axis=0).reshape(-1)

print("Transfoemer Feature shape:", X_train_tranf.shape)  # (N, 15)
print("CNN Features shape:", X_train_image.shape)
print("Target shape:", y_train.shape)

/tmp/ipykernel_31438/3680405253.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  temp = torch.tensor(temp, dtype=torch.float32).to(device)


Transfoemer Feature shape: (64214, 156)
CNN Features shape: (64214, 46)
Target shape: (64214,)


In [17]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [18]:
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,roc_auc_score, precision_score, recall_score, f1_score

In [19]:
tranf_params = {'n_estimators': 333, 'max_depth': 3, 'learning_rate': 0.0221721009745792, 'subsample': 0.7278504211411958, 'colsample_bytree': 0.7017750004514547, 'gamma': 3.687415269846868, 'min_child_weight': 4, 'reg_alpha': 0.3300301191023495, 'reg_lambda': 2.948137315140846}
xgb_tranf = XGBClassifier(**tranf_params)
xgb_tranf.fit(X_train_tranf,y_train)
y_pred_tranf = xgb_tranf.predict(X_val_tranf)

print("XGBoost Accuracy:", accuracy_score(y_val, y_pred_tranf))
print("XGBoost Precision:", precision_score(y_val, y_pred_tranf))
print("XGBoost Recall:", recall_score(y_val, y_pred_tranf))
print("XGBoost F1 Score:", f1_score(y_val, y_pred_tranf))
print("XGBoost ROC-AUC:", roc_auc_score(y_val, xgb_tranf.predict_proba(X_val_tranf)[:, 1]))

print("\nXGBoost Classification Report:\n")
print(classification_report(y_val, y_pred_tranf))

XGBoost Accuracy: 0.7632209792241755
XGBoost Precision: 0.8286621315192744
XGBoost Recall: 0.6636640999564144
XGBoost F1 Score: 0.7370416683473842
XGBoost ROC-AUC: 0.8410617785421548

XGBoost Classification Report:

              precision    recall  f1-score   support

         0.0       0.72      0.86      0.78     13766
         1.0       0.83      0.66      0.74     13766

    accuracy                           0.76     27532
   macro avg       0.77      0.76      0.76     27532
weighted avg       0.77      0.76      0.76     27532



In [20]:
import joblib
joblib.dump(xgb_tranf,"transformer_model_xgb.pkl")

['transformer_model_xgb.pkl']

In [21]:
image_xgb_params = {'n_estimators': 337, 'max_depth': 10, 'learning_rate': 0.10688764577817207, 'subsample': 0.7269736586402804, 'colsample_bytree': 0.921351603836381, 'gamma': 0.7690378444096395, 'min_child_weight': 1, 'reg_alpha': 4.077060755138467, 'reg_lambda': 4.466000832428299}

image_xgb = XGBClassifier(
    **image_xgb_params,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

image_xgb.fit(X_train_image, y_train)

# =========================
# FINAL EVALUATION (your format)
# =========================

y_pred_xgb = image_xgb.predict(X_val_image)

print("XGBoost Accuracy:", accuracy_score(y_val, y_pred_xgb))
print("XGBoost Precision:", precision_score(y_val, y_pred_xgb))
print("XGBoost Recall:", recall_score(y_val, y_pred_xgb))
print("XGBoost F1 Score:", f1_score(y_val, y_pred_xgb))
print("XGBoost ROC-AUC:", roc_auc_score(y_val, image_xgb.predict_proba(X_val_image)[:, 1]))

print("\nXGBoost Classification Report:\n")
print(classification_report(y_val, y_pred_xgb))

XGBoost Accuracy: 0.7563562400116228
XGBoost Precision: 0.7757893091591123
XGBoost Recall: 0.7211245096614848
XGBoost F1 Score: 0.7474587756946013
XGBoost ROC-AUC: 0.846228310262675

XGBoost Classification Report:

              precision    recall  f1-score   support

         0.0       0.74      0.79      0.76     13766
         1.0       0.78      0.72      0.75     13766

    accuracy                           0.76     27532
   macro avg       0.76      0.76      0.76     27532
weighted avg       0.76      0.76      0.76     27532



In [22]:
joblib.dump(image_xgb,"image_model_xgb.pkl")

['image_model_xgb.pkl']